[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C25_Long_Context_Course/05_kv_compression/05_kv_compression.ipynb)

# 05 · KV 压缩与长上下文评测（用 numpy 从零写出）

**KV 墙**：解码时 KV cache 随长度线性膨胀，吃显存又吃带宽。三条压缩路：**量化**(降比特)、**淘汰**(丢 token)、**合并**。
压缩是**近似**，必须用 **NIAH 大海捞针 / RULER 多跳** 检验「压缩后还看不看得见」。这是全课的收口。

**路线**：
1. KV cache 的增长（量级账）
2. **KV 量化**：INT8/INT4 仿射量化 + 误差测量
3. 分组量化 vs 全局量化（离群值）
4. **H2O 淘汰**：累积注意力分数 → heavy hitter + 最近窗口
5. **NIAH 大海捞针**：构造任务 + 召回测试 + lost-in-the-middle
6. **RULER 多跳**：变量追踪任务
7. ✏️ 练习（H2O 打分 / 量化误差 / NIAH 构造 / RULER 多跳）→ 📖 答案 → 🧪 胶囊

> 本课纪律：压缩方法都对拍「无损基准」量化损失；评测任务都有唯一答案、可自动判分。

## 1 · KV cache 随长度膨胀

KV cache = `2(K,V)·层数·KV头数·head_dim·n·字节`，**线性于 n**。算一笔账看它在长上下文下多大、压缩能省多少。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def kv_cache_gb(n, n_layers=32, n_kv_heads=32, head_dim=128, bits=16):
    return 2 * n_layers * n_kv_heads * head_dim * n * (bits/8) / 1e9

print(f"{'n':>8s} {'FP16 KV(GB)':>13s} {'INT4 KV(GB)':>13s} {'MQA+INT4(GB)':>14s}")
for n in [8192, 32768, 131072]:
    fp16 = kv_cache_gb(n)
    int4 = kv_cache_gb(n, bits=4)                      # 量化省 4×
    mqa4 = kv_cache_gb(n, n_kv_heads=1, bits=4)        # MQA(1头)+INT4
    print(f'{n:>8d} {fp16:>13.2f} {int4:>13.3f} {mqa4:>14.4f}')
assert abs(kv_cache_gb(8192, bits=4) - kv_cache_gb(8192)/4) < 1e-9, 'INT4 省 4×'
assert kv_cache_gb(131072, n_kv_heads=1) < kv_cache_gb(131072), 'MQA 省'
print('\n✅ KV 线性于 n；量化(比特)与 MQA(头数) 正交叠乘 → 128k 的 KV 从数十GB 压到 ~1GB')

## 2 · KV 量化：INT8/INT4 仿射量化

仿射量化：`q=round((x-z)/s)` 存整数，`x̂=s·q+z` 反量化。`s=(max-min)/(2^b-1)`, `z=min`。
比特 `b` 越小越省，误差越大。实现量化-反量化往返，测重构误差。

In [ ]:
def quantize_dequantize(x, bits=8):
    '''仿射量化往返：返回反量化后的近似 x̂。'''
    qmax = 2**bits - 1
    lo, hi = x.min(), x.max()
    s = (hi - lo) / qmax if hi > lo else 1.0
    q = np.round((x - lo) / s).clip(0, qmax)          # 量化为整数
    return s * q + lo                                  # 反量化

K = rng.standard_normal((64, 128))                    # 一组 K 向量
for bits in [8, 4, 2]:
    Kq = quantize_dequantize(K, bits)
    err = np.abs(K - Kq).mean()
    print(f'INT{bits}: 平均重构误差 = {err:.4f}')
err8 = np.abs(K - quantize_dequantize(K, 8)).mean()
err4 = np.abs(K - quantize_dequantize(K, 4)).mean()
assert err8 < err4, '比特越多误差越小'
assert np.allclose(K, quantize_dequantize(K, 16), atol=0.01), 'INT16 应几乎无损'
print('✅ 量化：比特越少越省显存、但误差越大（精度-显存权衡）')

## 3 · 分组量化 vs 全局量化（离群值）

若一组数有离群值，它撑宽量化范围 `[min,max]`、让正常值精度暴跌。**分组量化**(每小组各自量化)能隔离离群值。
构造带离群值的数据，对比全局 vs 分组量化的误差。

In [ ]:
def quant_global(x, bits=4):
    return quantize_dequantize(x, bits)

def quant_grouped(x, bits=4, group=16):
    '''按组量化：每 group 个元素各自一套 (s,z)。'''
    out = x.copy().astype(float)
    flat = out.reshape(-1)
    for g0 in range(0, len(flat), group):
        flat[g0:g0+group] = quantize_dequantize(flat[g0:g0+group], bits)
    return flat.reshape(x.shape)

x = rng.standard_normal(128)
x[7] = 50.0                                            # 一个离群值
err_global  = np.abs(x - quant_global(x, 4)).mean()
err_grouped = np.abs(x - quant_grouped(x, 4, group=16)).mean()
print(f'INT4 平均误差：全局量化={err_global:.4f}  分组量化={err_grouped:.4f}')
assert err_grouped < err_global, '分组量化应优于全局（隔离离群值）'
print('✅ 分组量化把离群值的污染限制在一组内 → 正常值精度大幅恢复（KIVI 等的关键）')

## 4 · H2O 淘汰：累积注意力分数 → heavy hitter

H2O：token 重要性 = 它被后续所有 query 关注的**累积注意力分数**。保留 heavy hitter + 最近窗口，淘汰其余。
**对拍**：H2O 保留的 token 应覆盖大部分注意力质量（vs 随机淘汰）。

In [ ]:
def h2o_importance(P):
    '''P:(n,n) 因果注意力概率。token j 的重要性 = 列 j 的累积(被多少后续 query 关注)。'''
    return P.sum(axis=0)                               # (n,) 每个 key 被关注的累积分数

def h2o_keep(P, budget, recent):
    '''返回保留的 token 下标：最近 recent 个 + 其余里 importance 最高的，凑满 budget。'''
    n = P.shape[0]
    imp = h2o_importance(P)
    recent_idx = set(range(n - recent, n))
    n_heavy = budget - len(recent_idx)
    cand = [j for j in range(n) if j not in recent_idx]
    heavy = sorted(cand, key=lambda j: imp[j], reverse=True)[:max(n_heavy, 0)]
    return sorted(recent_idx | set(heavy))

# 构造因果注意力，让少数 token 是 heavy hitter
n = 40
S = rng.standard_normal((n, n)); S[:, 2] += 6.0; S[:, 5] += 5.0    # token 2,5 是 heavy hitter
S = np.where(np.triu(np.ones((n,n), bool), 1), -np.inf, S)
P = np.exp(S - np.nanmax(np.where(np.isinf(S), -1e9, S), 1, keepdims=True))
P = np.where(np.isinf(S), 0.0, P); P /= P.sum(1, keepdims=True)

budget, recent = 12, 6
kept = h2o_keep(P, budget, recent)
mass_h2o = P[:, kept].sum() / P.sum()                 # H2O 保留覆盖的注意力质量
rng2 = np.random.default_rng(1)
rand_keep = sorted(rng2.choice(n, budget, replace=False))
mass_rand = P[:, rand_keep].sum() / P.sum()
print(f'保留 {budget}/{n} 个 token：H2O 覆盖注意力质量={mass_h2o:.3f}  随机={mass_rand:.3f}')
assert 2 in kept and 5 in kept, 'H2O 应保留 heavy hitter token 2,5'
assert mass_h2o > mass_rand, 'H2O 应比随机淘汰保住更多注意力质量'
print('✅ H2O：按累积注意力保留 heavy hitter + 最近窗口 → 小缓存维持大部分注意力质量')

## 5 · NIAH 大海捞针：构造 + 召回 + lost-in-the-middle

把一句「针」埋进长「干草堆」，测能否检索。我们用 token id 模拟：干草堆是随机 token，针是一个独特 token，
「检索」= 一个 query 能否注意到针的位置。扫描**深度**，复现 lost-in-the-middle。

In [ ]:
def build_niah(haystack_len, needle_depth, vocab=1000, needle_id=999):
    '''构造 NIAH：长度 haystack_len 的随机 token，在 depth 处插入独特 needle。返回 (序列, 针位置)。'''
    seq = rng.integers(0, vocab-1, size=haystack_len)   # 干草堆(不含 needle_id)
    pos = int(needle_depth * (haystack_len - 1))
    seq[pos] = needle_id                                  # 埋针
    return seq, pos

def retrieve_needle(seq, needle_id=999):
    '''模拟检索：找到 needle 的位置（真实模型靠注意力，这里直接定位以验证任务构造正确）。'''
    hits = np.where(seq == needle_id)[0]
    return int(hits[0]) if len(hits) else -1

for depth in [0.0, 0.5, 1.0]:
    seq, pos = build_niah(200, depth)
    found = retrieve_needle(seq)
    assert found == pos, f'针应在 depth={depth} 处, 位置 {pos}'
    print(f'depth={depth}: 针埋在位置 {pos}/200，检索到 {found} ✅')
assert (seq == 999).sum() == 1, '应恰好一根针'
print('✅ NIAH 任务构造正确：针可被精确定位（真实模型用注意力捞，这里验证任务本身无误）')

**lost-in-the-middle 模拟**：真实模型对中间深度检索差。用一个「检索强度随深度变化」的玩具模型展示 U 形曲线。

In [ ]:
def retrieval_score_toy(depth):
    '''玩具模型：开头/结尾检索好(≈1)、中间差 → U 形(lost in the middle)。'''
    return 0.4 + 0.6 * (2*depth - 1)**2                  # depth=0.5 最低

depths = np.linspace(0, 1, 11)
scores = np.array([retrieval_score_toy(d) for d in depths])
print('深度  : ' + ' '.join(f'{d:.1f}' for d in depths))
print('检索分: ' + ' '.join(f'{s:.2f}' for s in scores))
assert scores[5] == scores.min(), '中间(depth=0.5)检索最差'
assert scores[0] > scores[5] and scores[-1] > scores[5], '开头/结尾优于中间(U形)'
print('✅ lost-in-the-middle：开头/结尾检索好、中间差 → NIAH 必须扫深度，不能只在结尾放针')

## 6 · RULER 多跳：变量追踪

RULER 的变量追踪任务：`X1=val; X2=X1; X3=X2; ...` 问最后一个变量的值，需**串联多处分散信息**(多跳)。
比单针难得多——压缩/稀疏若断了任一环，多跳就失败。构造并求解这个任务。

In [ ]:
def build_variable_tracking(n_hops, haystack_len=100, vocab=10000):
    '''构造变量追踪链 X0=val; X1=X0; ...; 埋进干草堆。返回 (序列文本, 链, 最终答案)。'''
    val = int(rng.integers(1000, 9999))
    chain = [('X0', val)] + [(f'X{i}', f'X{i-1}') for i in range(1, n_hops)]
    # 把赋值语句打散插入随机位置（模拟分散在长文里）
    positions = sorted(rng.choice(haystack_len, n_hops, replace=False))
    doc = ['noise'] * haystack_len
    for (var, src), pos in zip(chain, positions):
        doc[pos] = f'{var}={src}'
    return doc, chain, val

def solve_variable_tracking(doc):
    '''顺着赋值链求解最终变量的值（需要找全所有跳）。'''
    env = {}
    # 多趟传播直到稳定（赋值可能乱序出现）
    stmts = [tok for tok in doc if tok != 'noise' and '=' in tok]
    for _ in range(len(stmts) + 1):
        for s in stmts:
            var, src = s.split('=')
            env[var] = int(src) if src.isdigit() else env.get(src)
    return env.get(f'X{len(stmts)-1}')

for hops in [2, 4, 6]:
    doc, chain, answer = build_variable_tracking(hops)
    got = solve_variable_tracking(doc)
    assert got == answer, f'{hops} 跳: 期望 {answer} 得到 {got}'
    print(f'{hops} 跳变量追踪: 答案={answer}, 求解={got} ✅')
print('✅ 多跳任务：需串联全部赋值才能答对；漏掉任一跳(被压缩/稀疏删掉)就断链 → 比单针严苛')

---
## ✏️ 练习 1：实现 H2O 重要性打分

实现 `importance_score(P)`：给定因果注意力概率 `P`(n,n)，返回每个 token 的重要性（被后续 query 关注的累积分数 = 列和）。
验证：被强烈关注的 token 重要性高。

In [ ]:
def importance_score(P):
    # TODO: 返回每列(每个 key token)的累积注意力 = P.sum(axis=0)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
n = 20
S = rng.standard_normal((n, n)); S[:, 3] += 8.0        # token 3 是 heavy hitter
S = np.where(np.triu(np.ones((n,n), bool), 1), -np.inf, S)
P = np.exp(S - np.nanmax(np.where(np.isinf(S),-1e9,S),1,keepdims=True))
P = np.where(np.isinf(S), 0.0, P); P /= P.sum(1, keepdims=True)
imp = importance_score(P)
assert imp.shape == (n,)
assert imp.argmax() == 3, 'heavy hitter token 3 应重要性最高'
assert np.allclose(imp.sum(), n, atol=1e-6) or imp.sum() > 0  # 总质量 = 行数
print(f'token 3 重要性={imp[3]:.2f}（最高），平均={imp.mean():.2f}')
print('✅ 练习 1 通过：H2O 累积注意力打分（heavy hitter 浮现）')

## ✏️ 练习 2：测量 KV 量化的下游误差

量化 K/V 会影响注意力输出。实现 `attn_quant_error(Q,K,V,bits)`：返回「量化 KV 算的注意力输出」与「FP16 的」的最大误差。
验证：比特越少误差越大。

In [ ]:
def attn_quant_error(Q, K, V, bits):
    def attn(Q, K, V):
        S = Q @ K.T / np.sqrt(Q.shape[1])
        P = np.exp(S - S.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
        return P @ V
    O_fp16 = attn(Q, K, V)
    # TODO: 量化 K、V(用 quantize_dequantize)，算 O_quant=attn(Q,Kq,Vq)，返回 max|O_fp16-O_quant|
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
n, d = 16, 8
Q = rng.standard_normal((n,d)); K = rng.standard_normal((n,d)); V = rng.standard_normal((n,d))
e8 = attn_quant_error(Q, K, V, 8)
e4 = attn_quant_error(Q, K, V, 4)
e2 = attn_quant_error(Q, K, V, 2)
assert e8 < e4 < e2, '比特越少下游误差越大'
print(f'注意力输出误差：INT8={e8:.4f}  INT4={e4:.4f}  INT2={e2:.4f}')
print('✅ 练习 2 通过：量化误差通过注意力传到输出，低比特代价可量化')

## ✏️ 练习 3：构造 NIAH 任务

实现 `make_niah(L, depth, needle=999)`：长度 L 的随机 token 序列(不含 needle)，在相对深度 depth 处插入 needle。
返回 `(序列, 针位置)`。验证：针在正确深度、恰好一根、其余不是 needle。

In [ ]:
def make_niah(L, depth, needle=999, vocab=1000):
    # TODO: seq=随机 token(0..vocab-2); pos=int(depth*(L-1)); seq[pos]=needle; 返回 (seq,pos)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
for depth in [0.0, 0.3, 1.0]:
    seq, pos = make_niah(150, depth)
    assert pos == int(depth * 149), f'针深度错: {pos}'
    assert seq[pos] == 999 and (seq == 999).sum() == 1, '恰好一根针'
    assert len(seq) == 150
print('✅ 练习 3 通过：NIAH 任务构造（可控深度、唯一针）—— 长上下文体检的第一关')

## ✏️ 练习 4：RULER 多跳——求解变量追踪链

实现 `solve_chain(assignments)`：给定赋值列表如 `['X0=42','X1=X0','X2=X1']`(可能乱序)，返回最后一个变量的值。
需要顺着引用链解析(多跳)。验证：链越长仍能正确求解。

In [ ]:
def solve_chain(assignments):
    # TODO: 多趟传播解析赋值(数字直接赋值, 变量引用查表)，返回 X{n-1} 的值
    #   提示: env={}; 循环 len+1 趟; 每趟对每个 'var=src': src.isdigit()? int : env.get(src)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert solve_chain(['X0=42', 'X1=X0', 'X2=X1']) == 42
assert solve_chain(['X2=X1', 'X0=7', 'X1=X0']) == 7         # 乱序也要对
long_chain = ['X0=99'] + [f'X{i}=X{i-1}' for i in range(1, 8)]
assert solve_chain(long_chain) == 99, '8 跳仍应正确'
print('✅ 练习 4 通过：多跳变量追踪求解（漏任一跳就断链 → 考验长上下文组合能力）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def importance_score(P):
    return P.sum(axis=0)

In [ ]:
# 练习 2 参考答案
def attn_quant_error(Q, K, V, bits):
    def attn(Q, K, V):
        S = Q @ K.T / np.sqrt(Q.shape[1])
        P = np.exp(S - S.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
        return P @ V
    O_fp16 = attn(Q, K, V)
    O_quant = attn(Q, quantize_dequantize(K, bits), quantize_dequantize(V, bits))
    return np.abs(O_fp16 - O_quant).max()

In [ ]:
# 练习 3 参考答案
def make_niah(L, depth, needle=999, vocab=1000):
    seq = rng.integers(0, vocab-1, size=L)
    pos = int(depth * (L - 1))
    seq[pos] = needle
    return seq, pos

In [ ]:
# 练习 4 参考答案
def solve_chain(assignments):
    env = {}
    for _ in range(len(assignments) + 1):
        for a in assignments:
            var, src = a.split('=')
            env[var] = int(src) if src.isdigit() else env.get(src)
    return env.get(f'X{len(assignments)-1}')

---
## 🧪 真实数据胶囊：压缩率 × 有效上下文的权衡

压缩率单看没意义，要和「任务表现」一起看。用一个玩具模型模拟：KV 淘汰预算越小，多跳任务越容易断链。
算不同压缩率下「能支持的最大跳数」，体会「压缩率」与「有效上下文」的真实权衡。

In [ ]:
def max_hops_under_budget(total_tokens, keep_ratio, hop_spacing=10):
    # TODO: 保留 keep_ratio 比例的 token；多跳链每跳间隔 hop_spacing 个 token；
    #   能完整保留的最大跳数 ≈ floor(total_tokens*keep_ratio / hop_spacing)
    #   返回该最大跳数
    raise NotImplementedError

In [ ]:
# 自测
total = 1000
print(f"{'压缩(keep)':>12s} {'保留token':>10s} {'可支持最大跳数':>16s}")
for ratio in [1.0, 0.5, 0.2, 0.1]:
    hops = max_hops_under_budget(total, ratio)
    print(f'{ratio:>12.0%} {int(total*ratio):>10d} {hops:>16d}')
assert max_hops_under_budget(total, 1.0) > max_hops_under_budget(total, 0.1), '压得越狠支持的跳数越少'
assert max_hops_under_budget(total, 0.5) == 50, '500 token / 间隔 10 = 50 跳'
print('\n✅ 压缩率越高 → 能保留的分散线索越少 → 多跳能力越早断链。')
print('   这就是为什么压缩必须用 RULER 多跳评测：单看「省了多少显存」会掩盖「丢了多少能力」。')

In [ ]:
# 📖 胶囊参考答案
def max_hops_under_budget(total_tokens, keep_ratio, hop_spacing=10):
    return int(total_tokens * keep_ratio) // hop_spacing

### 小结
- **KV 墙**：解码时 KV cache 随 n 线性膨胀，吃显存又吃带宽（访存受限）。与内存墙(prefill 的 n×n)正交。
- 三条压缩路：**量化**(降比特, 通用低风险, 注意 K/V 不对称 + 离群值→分组)、**淘汰**(丢 token)、**合并**。GQA/MQA 是结构性压缩(C20)。
- **H2O**：token 重要性 = 累积注意力分数；保留 heavy hitter + 最近窗口 → 小缓存维持质量（= StreamingLLM 的数据驱动版）。
- **评测是收口**：压缩/外推/稀疏/线性都是近似，必须用 **NIAH**(扫深度, 防 lost-in-the-middle) + **RULER**(多跳, 防断链) 检验「能看长是否=看得见」。
- **真实系统** = 按四道墙叠加正交杠杆，先吃免费的(FlashAttention/GQA)、再按需上有损的(稀疏/线性/KV淘汰)，每步用评测确认收益>代价。

🎉 **恭喜你读完并写完了整门课**。你现在掌握了长上下文的完整配方：四道墙、四条正交杠杆、一把验真的尺子。
下一步：把这些验证过的机制对应到真实框架(transformers 的 rope_scaling、flash-attn、KIVI、vLLM 的 PagedAttention)，接 C24 推理服务。